# V12 Formal Case-Adaptive Tail-Gain Calibration

## Why V12 is needed

The frozen V11 formula uses one global tail gain, `lambda = 0.90`. Its audit
showed that different cases can fail in opposite directions: increasing one
global gain may reduce P99 underprediction in one case while increasing the
maximum-stress overshoot in another. V12 tests whether correction strength can
instead be predicted from the input fields available at deployment.

The frozen stress structure is unchanged:

`sigma = sigma_V10 + delta_mu_case + lambda_case * scale_V10 * centered_tail_V11`

Only `lambda_case` is modelled. It is bounded to `[0, 1.10]` and may use only
case-level summaries of coordinates, FluenceRate, Temperature,
WeightLossRate, and the already-frozen predictor-only formula components.

## Statistical contract

- The 50 final-test case files remain sealed and are not read.
- All 149 non-final cases are development data because the former 15-case
  internal set was already inspected while V11 was designed.
- Similarity groups remain together in five outer and four inner folds.
- Each development case receives one nested out-of-fold gain prediction.
- These are **gain-layer OOF** results, not full-pipeline OOF results, because
  the frozen V11 base formula was originally developed on 119 cases.
- P95 and P99 are response-distribution evaluation metrics, not confidence
  levels, clipping boundaries, or deployment inputs.
- If any formal promotion gate fails, the saved deployment formula falls back
  automatically to the audited V11 gain `0.90`.


## 1. Imports and package root


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


def resolve_package_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "v12_case_adaptive_tail_gain.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate the NotebookCT3 package root.")


PACKAGE_ROOT = resolve_package_root()
if str(PACKAGE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT / "src"))

from v12_case_adaptive_tail_gain import (
    V12Config,
    candidate_configurations,
    output_directory,
    preflight_v12,
    run_v12,
)

print("Package root:", PACKAGE_ROOT)
print("Python:", sys.executable)


Package root: /net/scratch/j96317yn/NotebookCT3
Python: /net/scratch/j96317yn/NotebookCT3/.venv/bin/python


## 2. Locked formal configuration


In [2]:
RUN_V12 = True

CONFIG = V12Config(
    iteration=1,
    output_subdir="iteration_1",
    global_reference_gain=0.90,
    oracle_gain_min=0.00,
    oracle_gain_max=1.10,
    oracle_gain_step=0.025,
    oracle_near_optimal_tolerance=0.005,
    oracle_huber_delta=1.0,
    oracle_tier_mass=(0.55, 0.15, 0.20, 0.10),
    outer_folds=5,
    inner_folds=4,
    ridge_alphas=(0.1, 1.0, 10.0, 100.0),
    huber_alphas=(0.001, 0.01, 0.1),
    huber_epsilon=1.35,
    shrinkage_values=(0.50, 0.75, 1.00),
    minimum_deployable_gain=0.00,
    maximum_deployable_gain=1.10,
    bootstrap_resamples=10_000,
    random_seed=42,
)

OUTPUT_DIR = output_directory(PACKAGE_ROOT, CONFIG)
display(pd.DataFrame([CONFIG.__dict__]).T.rename(columns={0: "value"}))
display(candidate_configurations(CONFIG))
print("Output directory:", OUTPUT_DIR)


,value
iteration,1
output_subdir,iteration_1
global_reference_gain,0.9
oracle_gain_min,0.0
oracle_gain_max,1.1
oracle_gain_step,0.025
oracle_near_optimal_tolerance,0.005
oracle_huber_delta,1.0
oracle_tier_mass,"(0.55, 0.15, 0.2, 0.1)"
outer_folds,5


,candidate_id,model_family,feature_set,alpha,shrinkage,n_features,model_complexity
0,constant_global_0_90,constant,none,0.000,0.00,0,0
1,ridge__physical_9__a0.1__s0.50,ridge,physical_9,0.100,0.50,9,10
2,ridge__physical_9__a0.1__s0.75,ridge,physical_9,0.100,0.75,9,10
3,ridge__physical_9__a0.1__s1.00,ridge,physical_9,0.100,1.00,9,10
4,ridge__physical_9__a1__s0.50,ridge,physical_9,1.000,0.50,9,10
5,ridge__physical_9__a1__s0.75,ridge,physical_9,1.000,0.75,9,10
6,ridge__physical_9__a1__s1.00,ridge,physical_9,1.000,1.00,9,10
7,ridge__physical_9__a10__s0.50,ridge,physical_9,10.000,0.50,9,10
8,ridge__physical_9__a10__s0.75,ridge,physical_9,10.000,0.75,9,10
9,ridge__physical_9__a10__s1.00,ridge,physical_9,10.000,1.00,9,10


Output directory: /net/scratch/j96317yn/NotebookCT3/outputs/13_v12_case_adaptive_tail_gain/iteration_1


## 3. Frozen provenance and final-test seal

This cell verifies the exact V11 formula, V11 gain audit, 199-case manifest,
149-case development summary, similarity groups and V12 source hash. It checks
that the parent run read zero final-test cases and refuses to reuse an output
directory whose signature differs.


In [3]:
PREFLIGHT = preflight_v12(PACKAGE_ROOT, CONFIG)
display(PREFLIGHT["checks"])
display(PREFLIGHT["hash_audit"])
print("Development cases:", len(PREFLIGHT["development_ids"]))
print("Sealed final cases:", len(PREFLIGHT["inputs"]["final_ids"]))
print("V12 signature:", PREFLIGHT["signature"]["v12_signature_sha256"])


,check,value,expected,pass
0,development_cases,149,149,True
1,sealed_final_cases,50,50,True
2,v11_gain_audit_complete,complete,complete,True
3,parent_final_cases_read,0,0,True
4,frozen_global_gain,0.9,0.9,True
5,oracle_grid_values,45,45,True
6,context_feature_sets,2,2,True


,artifact,path,size_bytes,sha256
0,completion,/net/scratch/j96317yn/NotebookCT3/outputs/12_v...,708,8e83e59552f1c6c0ec0ab5e1e89107f5fc106f8f9169bc...
1,selected_gain,/net/scratch/j96317yn/NotebookCT3/outputs/12_v...,278,fdcf185a05b12212c3b1921ea392919bb0d6e8721dfcb4...
2,selected_formula,/net/scratch/j96317yn/NotebookCT3/outputs/12_v...,5570,1c254625ab6818c8f55c4f2edaf0717e6a26b89744d974...
3,parent_signature,/net/scratch/j96317yn/NotebookCT3/outputs/12_v...,2328,c0ce9ae6a5544e0d0d7548843d55c1894705046adefc15...
4,manifest,/net/scratch/j96317yn/NotebookCT3/shared/froze...,25685,8fa7fe2818b2d3bf69f47f5d214d033d84439bf22a7b78...
5,similarity_map,/net/scratch/j96317yn/NotebookCT3/shared/froze...,4408,2a3a8925ad9a6f6a0d107d974552e781f493e13b51dc95...
6,development_summary,/net/scratch/j96317yn/NotebookCT3/outputs/00_q...,132523,c0004723a438ee2bd7d5eb03123caae2640a9f51f45d27...
7,v12_source,/net/scratch/j96317yn/NotebookCT3/src/v12_case...,59803,f00b678f493fadafb80c72a6fec0feb3b9680a13ef8dfd...


Development cases: 149
Sealed final cases: 50
V12 signature: ec47276df49b70d2b8ed70c74d311c463ee0ee3bc2525fe26bfedf2f3fb467ed


## 4. Run or resume V12

The run is recoverable at case level. First, every development case is scanned
over 45 gains to define a tail-aware Huber oracle target. Then the bounded
Ridge/Huber gain layer is selected by nested similarity-group validation.
Finally, each OOF gain is checked against all 400,360 elements in its case.

Oracle gains use stress only as supervised development targets. The deployable
gain formula never receives stress or a stress-derived summary.


In [4]:
RESULT = None
if RUN_V12:
    RESULT = run_v12(PACKAGE_ROOT, CONFIG, preflight=PREFLIGHT)
    display(pd.DataFrame([RESULT]).T.rename(columns={0: "value"}))
else:
    print("V12 skipped because RUN_V12=False.")


[1/149] V12 oracle gain grid: case_01


[2/149] V12 oracle gain grid: case_06


[3/149] V12 oracle gain grid: case_10


[4/149] V12 oracle gain grid: case_100


[5/149] V12 oracle gain grid: case_101


...

[146/149] V12 complete-case OOF: case_96 gain=0.9413


[147/149] V12 complete-case OOF: case_97 gain=1.0044


[148/149] V12 complete-case OOF: case_98 gain=1.0086


[149/149] V12 complete-case OOF: case_99 gain=0.9234


,value
status,complete
iteration,1
method,bounded_case_adaptive_tail_gain_with_nested_si...
v12_signature_sha256,ec47276df49b70d2b8ed70c74d311c463ee0ee3bc2525f...
development_cases,149
nested_outer_folds,5
nested_inner_folds,4
gain_candidates,43
oracle_grid_values_per_case,45
selected_refit_candidate,huber__physical_9__a0.001__s1.00


## 5. Saved evidence and formal decision


In [5]:
artifacts = {
    "completion": OUTPUT_DIR / "v12_complete.json",
    "decision": OUTPUT_DIR / "v12_promotion_decision.json",
    "gates": OUTPUT_DIR / "v12_promotion_gates.csv",
    "scope_metrics": OUTPUT_DIR / "v12_comparison_scope_metrics.csv",
    "oof_gains": OUTPUT_DIR / "gain_layer_nested_oof_predictions.csv",
    "outer_choices": OUTPUT_DIR / "nested_outer_selected_candidates.csv",
    "refit_model": OUTPUT_DIR / "gain_model_refit_on_149.json",
    "selected_formula": OUTPUT_DIR / "selected_deployment_formula.txt",
    "adaptive_candidate": OUTPUT_DIR / "v12_candidate_adaptive_formula.txt",
}
for name, path in artifacts.items():
    print(f"{name:20s} exists={path.exists()}  {path}")

if artifacts["decision"].exists():
    print("\nPromotion decision")
    print(artifacts["decision"].read_text(encoding="utf-8"))
if artifacts["gates"].exists():
    display(pd.read_csv(artifacts["gates"]))
if artifacts["scope_metrics"].exists():
    display(pd.read_csv(artifacts["scope_metrics"]))
if artifacts["outer_choices"].exists():
    display(pd.read_csv(artifacts["outer_choices"]))
if artifacts["selected_formula"].exists():
    print("\n" + artifacts["selected_formula"].read_text(encoding="utf-8"))


completion           exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/13_v12_case_adaptive_tail_gain/iteration_1/v12_complete.json
decision             exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/13_v12_case_adaptive_tail_gain/iteration_1/v12_promotion_decision.json
gates                exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/13_v12_case_adaptive_tail_gain/iteration_1/v12_promotion_gates.csv
scope_metrics        exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/13_v12_case_adaptive_tail_gain/iteration_1/v12_comparison_scope_metrics.csv
oof_gains            exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/13_v12_case_adaptive_tail_gain/iteration_1/gain_layer_nested_oof_predictions.csv
outer_choices        exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/13_v12_case_adaptive_tail_gain/iteration_1/nested_outer_selected_candidates.csv
refit_model          exists=True  /net/scratch/j96317yn/NotebookCT3/outputs/13_v12_case_adaptive_tail_gain/i

,gate,pass,criterion
0,finite_bounded_gain_predictions,True,"all gains finite and in [0, 1.10]"
1,gain_layer_oracle_skill,True,weighted oracle-loss ratio < 0.995
2,all_case_macro_rmse_improves,False,all-149 macro RMSE improves by at least 0.5%
3,legacy_holdout_macro_rmse_not_worse,True,legacy 30-case macro RMSE does not worsen
4,all_case_p99_underprediction,False,all-149 mean P99 underprediction <= global and...
5,legacy_holdout_p99_underprediction,True,legacy holdout mean P99 underprediction worsen...
6,all_case_top1_overlap,True,all-149 top-1% overlap decreases by no more th...
7,legacy_holdout_top1_overlap,True,legacy holdout top-1% overlap decreases by no ...
8,maximum_prediction_guardrail,True,worst absolute maximum ratio <= global and <= ...
9,limited_gain_saturation,True,no more than 10% of OOF gains at bounds


,evaluation_scope,model,n_cases,n_elements_evaluated,micro_mae,micro_rmse,micro_r2,macro_mae,macro_rmse,macro_r2,...,mean_top5_actual_rmse,mean_top5_actual_bias,mean_p95_relative_error,mean_p95_underprediction_fraction,mean_p99_relative_error,mean_p99_underprediction_fraction,mean_top5pct_hotspot_overlap,mean_top1pct_hotspot_overlap,mean_top1_recall_in_predicted_top5,max_prediction_abs_max_ratio
0,all_149_development,v11_global_gain_0_90,149,59653640,1.211217,1.724897,0.629014,1.211217,1.689141,0.582822,...,3.108216,-1.653916,0.042882,0.002932,0.130866,0.126891,0.578844,0.588132,0.880990,1.584976
1,all_149_development,v12_adaptive_gain_nested_oof,149,59653640,1.206835,1.712541,0.634309,1.206835,1.681473,0.587573,...,3.077035,-1.610163,0.049238,0.004729,0.122723,0.118908,0.577474,0.590468,0.881164,1.477603
2,all_149_development,v12_oracle_gain_non_deployable,149,59653640,1.207498,1.712576,0.634294,1.207498,1.681312,0.588055,...,3.069721,-1.611746,0.046418,0.003825,0.121375,0.118781,0.578139,0.591707,0.881573,1.469059
3,legacy_119_train,v11_global_gain_0_90,119,47642840,1.196211,1.703422,0.629085,1.196211,1.667568,0.590481,...,3.099323,-1.693312,0.038829,0.003359,0.135189,0.132781,0.578520,0.588004,0.880777,1.543172
4,legacy_119_train,v12_adaptive_gain_nested_oof,119,47642840,1.192047,1.690907,0.634515,1.192047,1.659991,0.595188,...,3.067383,-1.647945,0.045866,0.005470,0.126260,0.124318,0.577148,0.590825,0.880890,1.449007
5,legacy_119_train,v12_oracle_gain_non_deployable,119,47642840,1.193515,1.692016,0.634035,1.193515,1.660895,0.594755,...,3.060052,-1.640643,0.044096,0.004380,0.125068,0.123575,0.577334,0.592097,0.881121,1.418631
6,legacy_15_validation,v11_global_gain_0_90,15,6005400,1.228952,1.769763,0.652403,1.228952,1.731626,0.538226,...,2.990331,-1.396082,0.077506,0.001988,0.102731,0.088884,0.571915,0.600516,0.881652,1.131529
7,legacy_15_validation,v12_adaptive_gain_nested_oof,15,6005400,1.230536,1.771035,0.651904,1.230536,1.733675,0.535721,...,2.981806,-1.343088,0.087259,0.003577,0.100149,0.082971,0.569657,0.596986,0.880436,1.135216
8,legacy_15_validation,v12_oracle_gain_non_deployable,15,6005400,1.227313,1.766321,0.653754,1.227313,1.727715,0.542314,...,2.972817,-1.410495,0.074841,0.003246,0.096614,0.087562,0.573960,0.597219,0.882051,1.101240
9,legacy_15_internal_test,v11_global_gain_0_90,15,6005400,1.312537,1.844602,0.600115,1.312537,1.817803,0.566658,...,3.296655,-1.599205,0.040414,0.000485,0.124700,0.118171,0.588347,0.576757,0.882018,1.584976


,outer_fold,outer_train_cases,outer_holdout_cases,outer_train_groups,outer_holdout_groups,candidate_id,model_family,feature_set,alpha,shrinkage,n_features,model_complexity,cv_selection_score,mean_oracle_loss_ratio,weighted_gain_mae,p90_absolute_gain_error,gain_saturation_fraction,fit_failure
0,0,119,30,92,23,huber__physical_9__a0.1__s1.00,huber,physical_9,0.100,1.0,9,11,1.003432,0.991185,0.062912,0.115755,0.008403,NaN
1,1,119,30,92,23,ridge__physical_9__a0.1__s1.00,ridge,physical_9,0.100,1.0,9,10,1.000108,0.989719,0.054400,0.098976,0.000000,NaN
2,2,119,30,92,23,huber__physical_9__a0.1__s1.00,huber,physical_9,0.100,1.0,9,11,1.000627,0.989720,0.052412,0.109951,0.008403,NaN
3,3,120,29,93,22,huber__physical_plus_tail_13__a0.1__s1.00,huber,physical_plus_tail_13,0.100,1.0,13,15,1.001181,0.990115,0.053385,0.114543,0.000000,NaN
4,4,119,30,91,24,huber__physical_9__a0.001__s1.00,huber,physical_9,0.001,1.0,9,11,0.999324,0.989017,0.052684,0.100775,0.000000,NaN



V12 formally selected deployment formula

Decision: retain_v11_global_gain_0_90
lambda_case = 0.9

sigma = ((-0.0116301024532845*temperature_mean + 0.0470427355318997*temperature_p95 - 3.16087946558646*weight_loss_rate_mean + 0.103449215368151*z_max - 0.870195660324862*Abs(4.60760803334225*fluence_rate_p95 - 20.3572633494299) - 100.195330145459) + exp(-1.47091131932291*rho_mean - 1011.416708227*theta_sin_std - 1.1690770556861*weight_loss_rate_mean + 0.904578545888617*z_mean + 9.23869712445276) * ((1.10317833861391*(0.55673575*(0.0509611749551904*rho - 9.32241465638866)*(0.0509611749551904*rho - 8.66867535638866) - 31.9731949759449*(theta_cos - 0.922666019258146)**2)*(Abs(0.0509611749551904*rho - 8.59575422638866) - 2.4201684) + 0.285739561816426) + (0.0005298607274781943*z + 0.17657582192765694*(6.3559467282562*nearest_axial_boundary_fraction_proxy - 1.50570424883607)*(-7.28799975835273*nearest_radial_boundary_fraction_proxy + 0.0517386902634747*rho - 6.047025771972898) - 0.2433234792

## 6. How to interpret the result

Three models are reported over identical complete development cases:

1. `v11_global_gain_0_90`: the frozen V11 reference.
2. `v12_adaptive_gain_nested_oof`: each case's gain is predicted without using
   that case or any case in its similarity group for gain-layer fitting.
3. `v12_oracle_gain_non_deployable`: the best near-optimal grid gain found
   using the true case stress. This is an upper-bound diagnostic only.

If the oracle does not improve a case, gain adaptation cannot repair that
case's spatial shape and a new residual formula would be required. If the
oracle improves it but nested OOF does not, the available predictor context is
insufficient to generalise the required gain. Promotion requires improvement
in macro RMSE without degrading P99 underprediction, hotspot overlap or the
maximum-prediction guardrail. Failure leaves the selected formula at V11's
global gain `0.90`; it is a valid negative result, not a failed run.
